<a href="https://colab.research.google.com/github/jAgaThasweety/rabacademy/blob/main/rabacademy2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# DATA INGESTION, CLEANING & PREPROCESSING WITH PANDAS
# ============================================================

import pandas as pd
import numpy as np
from google.colab import files

# ------------------------------------------------------------
# 1. UPLOAD DATASET
# ------------------------------------------------------------

print("Upload your raw CSV file:")
uploaded = files.upload()

filename = list(uploaded.keys())[0]

df = pd.read_csv(filename)

print("\n" + "="*70)
print("DATASET LOADED")
print("="*70)

print("File:", filename)
print("Rows:", len(df))
print("Columns:", len(df.columns))

print("\nFirst 5 rows:")
display(df.head())


# ------------------------------------------------------------
# 2. BEFORE CLEANING PROFILE
# ------------------------------------------------------------

print("\n" + "="*70)
print("BEFORE CLEANING")
print("="*70)

print("\nDataset Shape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
missing_before = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": df.isnull().mean() * 100
})

display(missing_before)

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nNumerical Summary:")
display(df.describe(include="all").T)


# ------------------------------------------------------------
# 3. STANDARDIZE COLUMN NAMES
# ------------------------------------------------------------

df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print("\nStandardized column names:")
print(df.columns.tolist())


# ------------------------------------------------------------
# 4. REMOVE DUPLICATE RECORDS
# ------------------------------------------------------------

duplicates_before = df.duplicated().sum()

df = df.drop_duplicates()

duplicates_removed = duplicates_before - df.duplicated().sum()

print("\nDuplicates removed:", duplicates_removed)


# ------------------------------------------------------------
# 5. CONVERT OBJECT COLUMNS
# ------------------------------------------------------------

for col in df.columns:

    if df[col].dtype == "object":

        df[col] = df[col].astype(str).str.strip()

        # Replace common missing-value representations
        df[col] = df[col].replace(
            ["", "nan", "NaN", "NULL", "null", "N/A", "NA", "-"],
            np.nan
        )


# ------------------------------------------------------------
# 6. AUTOMATICALLY CONVERT NUMERIC COLUMNS
# ------------------------------------------------------------

for col in df.columns:

    if df[col].dtype == "object":

        converted = pd.to_numeric(
            df[col],
            errors="coerce"
        )

        # Convert only when most values are numeric
        valid_ratio = converted.notna().mean()

        if valid_ratio >= 0.8:
            df[col] = converted


# ------------------------------------------------------------
# 7. DETECT DATE COLUMNS
# ------------------------------------------------------------

date_columns = []

for col in df.columns:

    if (
        "date" in col.lower()
        or "time" in col.lower()
    ):

        converted_date = pd.to_datetime(
            df[col],
            errors="coerce"
        )

        valid_ratio = converted_date.notna().mean()

        if valid_ratio >= 0.5:

            df[col] = converted_date
            date_columns.append(col)


print("\nDetected date columns:")
print(date_columns)


# ------------------------------------------------------------
# 8. HANDLE MISSING VALUES
# ------------------------------------------------------------

for col in df.columns:

    missing = df[col].isnull().sum()

    if missing == 0:
        continue

    # Numerical columns → median
    if pd.api.types.is_numeric_dtype(df[col]):

        median_value = df[col].median()

        if pd.notna(median_value):
            df[col] = df[col].fillna(median_value)

    # Categorical columns → mode
    elif df[col].dtype == "object":

        mode = df[col].mode()

        if len(mode) > 0:
            df[col] = df[col].fillna(mode[0])


# ------------------------------------------------------------
# 9. HANDLE REMAINING MISSING VALUES
# ------------------------------------------------------------

for col in df.columns:

    if df[col].isnull().sum() > 0:

        if df[col].dtype == "object":
            df[col] = df[col].fillna("Unknown")

        elif pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(0)


# ------------------------------------------------------------
# 10. HANDLE OUTLIERS USING IQR
# ------------------------------------------------------------

numeric_columns = df.select_dtypes(
    include=np.number
).columns

outlier_summary = []

for col in numeric_columns:

    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = (
        (df[col] < lower_bound) |
        (df[col] > upper_bound)
    )

    count = outliers.sum()

    outlier_summary.append({
        "Column": col,
        "Outliers": count,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound
    })

    # Cap extreme values instead of deleting records
    df[col] = df[col].clip(
        lower=lower_bound,
        upper=upper_bound
    )


outlier_report = pd.DataFrame(outlier_summary)

print("\n" + "="*70)
print("OUTLIER REPORT")
print("="*70)

display(outlier_report)


# ------------------------------------------------------------
# 11. FEATURE ENGINEERING - DATE FEATURES
# ------------------------------------------------------------

for col in date_columns:

    df[f"{col}_year"] = df[col].dt.year
    df[f"{col}_month"] = df[col].dt.month
    df[f"{col}_month_name"] = df[col].dt.month_name()
    df[f"{col}_quarter"] = df[col].dt.quarter


# ------------------------------------------------------------
# 12. DETECT SALES / PRICE / QUANTITY COLUMNS
# ------------------------------------------------------------

def find_column(names):

    for name in names:

        if name in df.columns:
            return name

    return None


price_col = find_column([
    "price",
    "unit_price",
    "price_per_unit",
    "selling_price"
])

quantity_col = find_column([
    "quantity",
    "qty",
    "units",
    "units_sold"
])

sales_col = find_column([
    "sales",
    "revenue",
    "total",
    "total_amount",
    "total_spent"
])

cost_col = find_column([
    "cost",
    "cost_price",
    "unit_cost"
])


print("\nDetected business columns:")
print("Price    :", price_col)
print("Quantity :", quantity_col)
print("Sales    :", sales_col)
print("Cost     :", cost_col)


# ------------------------------------------------------------
# 13. FEATURE ENGINEERING - SALES
# ------------------------------------------------------------

if (
    price_col is not None
    and quantity_col is not None
):

    df["calculated_sales"] = (
        df[price_col] *
        df[quantity_col]
    )


# ------------------------------------------------------------
# 14. FEATURE ENGINEERING - PROFIT
# ------------------------------------------------------------

if (
    sales_col is not None
    and cost_col is not None
):

    df["profit"] = (
        df[sales_col] -
        df[cost_col]
    )

    df["profit_margin"] = np.where(
        df[sales_col] != 0,
        (df["profit"] / df[sales_col]) * 100,
        0
    )


# If unit price and cost are available
elif (
    price_col is not None
    and cost_col is not None
):

    df["profit_per_unit"] = (
        df[price_col] -
        df[cost_col]
    )

    if quantity_col is not None:

        df["profit"] = (
            df["profit_per_unit"] *
            df[quantity_col]
        )


# ------------------------------------------------------------
# 15. CLEAN TEXT VALUES
# ------------------------------------------------------------

for col in df.select_dtypes(
    include="object"
).columns:

    df[col] = (
        df[col]
        .str.strip()
        .str.replace(
            r"\s+",
            " ",
            regex=True
        )
    )


# ------------------------------------------------------------
# 16. FINAL MISSING VALUE CHECK
# ------------------------------------------------------------

missing_after = pd.DataFrame({

    "Missing Count": df.isnull().sum(),

    "Missing %": (
        df.isnull().mean() * 100
    )
})


# ------------------------------------------------------------
# 17. AFTER CLEANING PROFILE
# ------------------------------------------------------------

print("\n" + "="*70)
print("AFTER CLEANING")
print("="*70)

print("\nFinal Shape:")
print(df.shape)

print("\nRemaining Missing Values:")
display(
    missing_after[
        missing_after["Missing Count"] > 0
    ]
)

print("\nRemaining Duplicate Rows:")
print(df.duplicated().sum())

print("\nFinal Data Types:")
print(df.dtypes)

print("\nFinal Dataset Preview:")
display(df.head())


# ------------------------------------------------------------
# 18. BEFORE vs AFTER SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame({

    "Metric": [
        "Rows",
        "Columns",
        "Duplicate Rows",
        "Missing Cells"
    ],

    "Before Cleaning": [
        len(pd.read_csv(filename)),
        len(pd.read_csv(filename).columns),
        pd.read_csv(filename).duplicated().sum(),
        pd.read_csv(filename).isnull().sum().sum()
    ],

    "After Cleaning": [
        len(df),
        len(df.columns),
        df.duplicated().sum(),
        df.isnull().sum().sum()
    ]
})

print("\n" + "="*70)
print("BEFORE vs AFTER CLEANING")
print("="*70)

display(summary)


# ------------------------------------------------------------
# 19. EXPORT CLEAN DATASET
# ------------------------------------------------------------

output_file = "clean_dataset.csv"

df.to_csv(
    output_file,
    index=False
)

print("\n" + "="*70)
print("PROJECT COMPLETED")
print("="*70)

print("\nClean dataset created:")
print(output_file)

print("\nFinal rows:", len(df))
print("Final columns:", len(df.columns))

print("\nDownload starting...")

files.download(output_file)

Upload your raw CSV file:


Saving retail-orders-raw.csv to retail-orders-raw (1).csv

DATASET LOADED
File: retail-orders-raw (1).csv
Rows: 12
Columns: 9

First 5 rows:


,order_id,order_date,customer_segment,city,category,quantity,unit_price,discount_pct,payment_status
0,RT-1001,2026-01-03,Student,Chennai,Learning Kit,2,799,10.0,Paid
1,RT-1002,03/01/2026,Fresher,Bengaluru,Course Access,1,1499,0.0,paid
2,RT-1003,2026-01-05,student,Chennai,Course Access,1,1499,NaN,Pending
3,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid
4,RT-1004,2026-01-07,Professional,Hyderabad,Learning Kit,3,799,5.0,Paid



BEFORE CLEANING

Dataset Shape:
(12, 9)

Data Types:
order_id             object
order_date           object
customer_segment     object
city                 object
category             object
quantity             object
unit_price            int64
discount_pct        float64
payment_status       object
dtype: object

Missing Values:


,Missing Count,Missing %
order_id,0,0.000000
order_date,1,8.333333
customer_segment,0,0.000000
city,1,8.333333
category,0,0.000000
quantity,0,0.000000
unit_price,0,0.000000
discount_pct,1,8.333333
payment_status,0,0.000000


...


BEFORE vs AFTER CLEANING


,Metric,Before Cleaning,After Cleaning
0,Rows,12,11
1,Columns,9,14
2,Duplicate Rows,1,0
3,Missing Cells,3,15



PROJECT COMPLETED

Clean dataset created:
clean_dataset.csv

Final rows: 11
Final columns: 14

Download starting...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>